In [0]:
#set file paths here
path_employees   = 'paste_file_path_here'
path_departments = 'paste_file_path_here'
path_sales       = 'paste_file_path_here'

In [0]:
# import necessary libraries

from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from pyspark.sql.functions import col, to_date, avg, max as max_s, min as min_s, count, rank, dense_rank, row_number
from pyspark.sql.window import Window

**BASIC PYSPARK**

**Q1**
Read Employees as a DataFrame.

printSchema()

show()

inferSchema=False then manually define schema.


In [0]:
employees_df = spark.read\
                    .format('csv')\
                    .option('header','true')\
                    .load(path_employees)

employees_df.printSchema()
employees_df.show()

employees_schema = StructType([StructField('emp_id'    , IntegerType(), nullable = True, metadata = None)
                              ,StructField('name'      , StringType() , nullable = True, metadata = None)
                              ,StructField('dept_id'   , IntegerType(), nullable = True, metadata = None)
                              ,StructField('salary'    , StringType() , nullable = True, metadata = None)
                              ,StructField('city'      , StringType() , nullable = True, metadata = None)
                              ,StructField('join_date' , StringType() , nullable = True, metadata = None)
                              ,StructField('manager_id', IntegerType(), nullable = True, metadata = None)
                   ])
employees_df = spark.read\
                    .format('csv')\
                    .option('header','true')\
                    .option('inferSchema','false')\
                    .schema(employees_schema)\
                    .load(path_employees)
# DateType() for join_date is not able to read the schema and is showing up nulls in the column, it also doesn't any format parameter, its only useful when you use a parquet file as a source
# i am explicitly defining a column with date datatype instead of string, THIS STEP CAN BE DONE LATER though
employees_df = employees_df.withColumn('join_date'
                                       ,to_date(col('join_date'),'dd-MM-yyyy')
                            )
display(employees_df)


------------------------------------------------------------------------------------------------------------------

**Q2**
Select

emp_id
name
salary

only.

In [0]:
employees_df_q2 = employees_df.select('emp_id'
                                     ,'name'
                                     ,'salary'
                               )
display(employees_df_q2)

**Q3**
Filter

Salary > 80,000

In [0]:
employees_df_q3 = employees_df.filter(col("salary")>80000
                               )
display(employees_df_q3)

**Q4**
Find employees

Chicago
AND
salary > 70000

In [0]:
employees_df_q4 = employees_df.filter(  (col('city') == 'Chicago')
                                      & (col('salary') > 70000)
                               )

display(employees_df_q4)

**Q5**
Create

Annual Salary

salary * 12

In [0]:
employees_df_q5 = employees_df.withColumn('annual_salary', col('salary') * 12 
                               )
display(employees_df_q5)

**Q6**
Rename

manager_id to manager

In [0]:
employees_df_q6 = employees_df.withColumnRenamed('manager_id','manager')
display(employees_df_q6)

**AGGREGATIONS**

**Q7**
Department wise

Average salary

In [0]:
employees_df_q7 = employees_df.groupBy(col("dept_id")).agg(avg(col('salary')))

display(employees_df_q7)

**Q8**
Department wise

Maximum salary

Minimum salary

Average salary

Employee count

In [0]:
employees_df_q8 = employees_df.groupBy(col('dept_id')).agg(max_s(col('salary'))
                                                          ,min_s(col('salary'))
                                                          ,avg(col('salary'))
                                                          ,count(col('emp_id'))
                                                       )
display(employees_df_q8)

**Q9**
Find top 3 highest paid employees.

In [0]:
employees_df_q9 = employees_df.select('emp_id').orderBy(col('salary').desc()).limit(3)
display(employees_df_q9)

**Q10**
Count employees city-wise.

In [0]:
employees_df_q10 = employees_df.groupBy(col('city')).agg(count(col('emp_id')))
display(employees_df_q10)

**Q11**
Find departments having Average salary > 75,000

In [0]:
employees_df_q11 = employees_df.groupBy(col('dept_id')).agg(avg(col('salary')).alias('average_salary')).filter(col('average_salary')>75000)
display(employees_df_q11)

**JOINS**

**Q12**
Join Employees with Departments.

In [0]:
departments_df = spark.read.format('csv')\
                           .option('header', 'true')\
                           .load(path_departments)

display(departments_df)

emp_dept_joined_df_q12 = employees_df.join(departments_df
                                          ,employees_df.dept_id == departments_df.dept_id
                                          ,'left'
                                      )\
                                     .drop(departments_df.dept_id) #to remove an extra dept_id column coming from departments dataframe.
display(emp_dept_joined_df_q12)

**Q13** Find employees without matching department.

In [0]:
emp_dept_joined_df_q13 = employees_df.join(departments_df
                                          ,employees_df.dept_id == departments_df.dept_id
                                          ,'left_anti'
                                      )\
                                     .drop(departments_df.dept_id)    
display(emp_dept_joined_df_q13) #all employees have matching employees, no rows returned

**Q14** Find departments without employees.

In [0]:
emp_dept_joined_df_q14 = employees_df.join(departments_df
                                          ,employees_df.dept_id == departments_df.dept_id
                                          ,'left'  
                                      )\
                                     .drop(departments_df.dept_id)
emp_dept_joined_df_q14 = emp_dept_joined_df_q14.groupBy(col('dept_name')).agg(count(col('emp_id')).alias('count_of_employees'))

display(emp_dept_joined_df_q14.filter(col('count_of_employees')==0)) #all departments have atleast 1 employee, no rows returned

**WINDOW FUNCTIONS**

**Q16**
Rank employees by salary.
Use

rank

dense_rank

row_number

In [0]:
emp_df_ranked_df_q17 = employees_df.withColumn('rank_col'
                                              ,rank().over(Window.orderBy(col('salary').desc()))
                                    )
display(emp_df_ranked_df_q17)

emp_df_dense_ranked_df_q17 = employees_df.withColumn('rank_col'
                                                    ,rank().over(Window.orderBy(col('salary').desc()))
                                          )
display(emp_df_dense_ranked_df_q17)

emp_df_row_numbered_df_q17 = employees_df.withColumn('rank_col'
                                                    ,row_number().over(Window.orderBy(col('salary').desc()))
                                          )
display(emp_df_row_numbered_df_q17)

**Q17**
Find highest salary employee in each department.